# 02 – Bronze → Silver Transformation (`orders`)

**Purpose:** turn the three raw daily order files (Bronze) into a typed, standardized Silver `orders` table, driven by the issues in the DQ report.

**Inputs:** `data/orders/orders_2026-10-05/06/07.csv` (read-only). **Outputs** (folder `output/`):

| File | Content |
|---|---|
| `silver_orders.csv` | Business columns only (17 source columns, cleaned and typed) + `_source_file`, `_source_line` as the trace key |
| `silver_orders_dq_flags.csv` | One row per Silver row: the trace key, `dq_*` flags, and the raw value of every column we changed |
| `quarantine_orders.csv` | Lines that are **not** in Silver, with the raw line and the reason |

**Principles used throughout:** never invent values (no zero-fill, no made-up IDs, no sign flips); keep the raw value next to anything we null out; a DQ failure is first a *flag*, and only becomes a quarantine when the record cannot be used safely.

> The requirements discussion with Pratik was not part of the files I received, so decisions below rest on the DQ report and the data only. Anything that needs his confirmation is marked **[Unconfirmed]** and collected in the last section.

In [1]:
import csv, re
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR, OUT_DIR = Path("data/orders"), Path("output")
OUT_DIR.mkdir(exist_ok=True)

COLS = ["order_id","customer_id","customer_name","email","phone","city","order_date","ship_date",
        "status","quantity","unit_price","amount","currency","region_code","payment_mode","notes","modified_at"]
ALLOWED_STATUS = ["PLACED","SHIPPED","DELIVERED","CANCELLED","RETURNED"]   # allowed set from the DQ report
NULL_TOKENS = {"", "null", "n/a", "nan", "-", "none", "?"}                   # placeholders seen in the data
quarantine = []   # every record that leaves the main flow lands here with a reason

## 1. Structural ingestion: malformed CSV lines

**Identify:** The files contain lines whose field count is not 17. The DQ report never lists this (its checks assume a clean grid), but the row total of 5,235 includes them. A plain `pd.read_csv` fails on them.

**Understand:** A line with the wrong field count shifts every later column. Loading it blindly would put a `status` value in the `quantity` column, or silently drop data.

**Assess:** Two shapes exist, 10 lines per file. (a) 18 lines have 19 fields: the 17 expected fields followed by two extra tokens `EXTRA,COLS`. (b) 12 lines have only 9 fields, ending after `status`. For shape (a), two columns that have a tightly controlled vocabulary, `currency` (always `INR`) and `payment_mode` (5 values), sit exactly where the header says they should. For shape (b), `quantity`, `unit_price`, `amount` and the rest are simply absent.

**Decide:** Repair shape (a) by keeping the first 17 fields, but only when the anchor columns `currency` and `payment_mode` are valid in their expected positions. Quarantine shape (b) with its raw line.

**Implement:** Parse with the `csv` module line by line, keep `_source_file`, `_source_line` and the raw line for traceability, mark repaired rows with `dq_repaired_extra_fields`, and append quarantined lines to `quarantine`.

**Justify:** *Repair everything* would invent values for shape (b) (a missing `amount` cannot be guessed). *Quarantine everything* throws away 18 complete, usable orders because of trailing junk. Repairing only when alignment is provable keeps information without guessing, and the dropped extra tokens stay visible in the raw line kept in the flags file.

In [2]:
records = []
for f in sorted(DATA_DIR.glob("orders_*.csv")):
    lines = f.read_text(encoding="utf-8-sig").splitlines()
    assert next(csv.reader([lines[0]])) == COLS, f"unexpected header in {f.name}"
    for n, line in enumerate(lines[1:], start=2):
        r = next(csv.reader([line]))
        records.append(dict(zip(COLS, r[:17]), _n_fields=len(r), _source_file=f.name, _source_line=n, _raw_line=line))
raw_all = pd.DataFrame(records)
total_lines = len(raw_all)

# Anchor values are learned from well-formed rows, not hard-coded
wf = raw_all[raw_all._n_fields == 17]
anchor_currency, anchor_pay = set(wf.currency), set(wf.payment_mode)

too_short = raw_all._n_fields < 17
too_long  = raw_all._n_fields > 17
aligned   = too_long & raw_all.currency.isin(anchor_currency) & raw_all.payment_mode.isin(anchor_pay)
unsafe    = too_short | (too_long & ~aligned)

for _, r in raw_all[unsafe].iterrows():
    quarantine.append(dict(_source_file=r._source_file, _source_line=r._source_line,
                           quarantine_reason=f"malformed_csv_{r._n_fields}_fields_not_safely_repairable", raw_line=r._raw_line))

df = raw_all[~unsafe].copy()
df["dq_repaired_extra_fields"] = df._n_fields > 17
print(f"lines={total_lines}  kept={len(df)}  repaired(19→17)={int(df.dq_repaired_extra_fields.sum())}  quarantined={int(unsafe.sum())}")

lines=5235  kept=5223  repaired(19→17)=18  quarantined=12


## 2. Exact duplicate rows

**Identify:** 75 rows are fully identical to another row (all 17 columns), both in the source and in Bronze (the load did not create them).

**Understand:** Exact copies double-count quantity and revenue in every aggregate and add nothing to any join.

**Assess:** The report counts 75 duplicates in source and in target, so they are present in the source data. An exact copy carries no extra information.

**Decide:** Keep the first occurrence and remove the rest, but do it on the **raw** values (before any cleaning), and write every removed copy to the quarantine file instead of deleting it silently.

**Implement:** `duplicated(subset=COLS)` on the 17 raw columns, across all three files.

**Justify:** Deduplicating on the raw text is the only way to be sure the rows really are identical; deduplicating after cleaning would also merge rows that merely *look* alike (e.g. `PLACED` vs `placed`), which may be different versions of an order. Keeping a copy in quarantine makes the removal auditable. Rows that share only an `order_id` are **not** handled here (see section 9).

In [3]:
dup_mask = df.duplicated(subset=COLS, keep="first")
for _, r in df[dup_mask].iterrows():
    quarantine.append(dict(_source_file=r._source_file, _source_line=r._source_line,
                           quarantine_reason="exact_duplicate_removed", raw_line=r._raw_line))
df = df[~dup_mask].copy()
print("exact duplicates removed:", int(dup_mask.sum()), "| rows left:", len(df))

exact duplicates removed: 75 | rows left: 5148


## 3. Whitespace and placeholder strings

**Identify:** Leading/trailing blanks and placeholder strings (`NULL`, `N/A`, `NaN`, `-`, `none`, `?`, blank) stand in for missing values in several columns (order_id, email, amount, order_date, status, modified_at).

**Understand:** Placeholders are text, so a numeric or date cast fails, a status filter misses them, and `'N/A'` can accidentally join to another `'N/A'`. Padding makes `' Kolkata'` and `'Kolkata'` different group keys.

**Assess:** The report treats them as type/format failures (e.g. `"NULL"` in `order_id`, `NaN`/`-`/`N/A` in `amount`). The set of tokens above is exactly what appears in the data.

**Decide:** Trim every text column and convert the placeholder tokens to a real null in the working copy. The raw values stay in `df_raw_values` so we can say later *what* the source contained.

**Implement:** One `strip` + token-to-null pass. A snapshot of the raw (trimmed-free) values is kept for the flags file.

**Justify:** Converting to null (instead of leaving `'N/A'`, or filling zero/unknown) lets later steps use normal null logic, and it never fabricates a value. The original text survives in the snapshot, so nothing is lost.

In [4]:
df_raw_values = df[COLS].add_prefix("raw_")          # untouched copy for traceability
for c in COLS:
    s = df[c].str.strip()
    df[c] = s.mask(s.str.lower().isin(NULL_TOKENS))   # placeholders → real null

## 4. Identifiers: `order_id` and `customer_id`

**Identify:** `order_id` is blank/`NULL` in 54 rows (report: 29 null + 25 not convertible to bigint, Block severity); 53 remain after exact duplicates are removed. `customer_id` is blank in 42 rows (0.80%, within the report's 2% tolerance).

**Understand:** `order_id` is the business key of the table: without it a row cannot be de-duplicated, merged or traced back to the order system. `customer_id` only matters for customer-level joins; an order without it is still a valid sale.

**Assess:** The report treats the two differently (Block vs. an allowed 2%), and the data supports that: `customer_id` values are `C####` strings, `order_id` values are 6-digit integers, and every non-matching value is blank.

**Decide:**
- `order_id`: cast to integer; rows where it is null are **quarantined** (not in Silver) with their raw line.
- `customer_id`: keep as the text code `C####`; blank stays null and gets `dq_missing_customer_id`. No ID is invented.

**Implement:** `Int64` cast for `order_id`; regex `^C\d+$` check for `customer_id`. The null-key rows are removed at the end of the pipeline (section 10) so earlier steps still see them.

**Justify:** Keeping a keyless row in Silver would put a record in the table that nobody can reconcile or update, and every downstream "count distinct orders" would be wrong; the information is not lost because the quarantine file holds the full raw line. For `customer_id`, dropping or inventing would both be worse than a null with a flag. **[Unconfirmed]** Whether keyless orders should instead be kept in Silver is a business call.

In [5]:
df["order_id"] = pd.to_numeric(df["order_id"], errors="coerce").astype("Int64")
df["dq_missing_order_id"] = df["order_id"].isna()

bad_cust = df.customer_id.notna() & ~df.customer_id.str.fullmatch(r"C\d+", na=False)
df.loc[bad_cust, "customer_id"] = pd.NA
df["dq_missing_customer_id"] = df["customer_id"].isna()
print("missing order_id:", int(df.dq_missing_order_id.sum()), "| missing customer_id:", int(df.dq_missing_customer_id.sum()))

missing order_id: 53 | missing customer_id: 42


## 5. Dates: mixed formats in `order_date`, `ship_date`, `modified_at`

**Identify:** `order_date` fails `yyyy-MM-dd` in 374 rows (report); `ship_date` and `modified_at` show the same problem. Formats seen: `2026-10-05`, `05/10/2026`, `2026/10/05`, `Oct 5 2026`, plus placeholders and impossible dates (`00-00-0000`, `2026-13-01`).

**Understand:** Dates drive daily partitioning, late-shipment analysis and the `ship_date ≥ order_date` rule. `05/10/2026` is ambiguous (5 Oct or 10 May), and a wrong guess silently moves an order by months.

**Assess:** We can test the convention on the data instead of assuming it. Each daily file carries its own date, and `order_date` in ISO rows equals that date. The check below measures how often the *day-first* reading of the slash/dash dates matches the file date.

**Decide:** Parse with an explicit list of formats, **day-first** for `dd/mm/yyyy` (assumption supported by the check below), and never use a guessing parser. Impossible dates (`00-00-0000`, month 13) and placeholders are not repaired: they become null with a flag, and the raw text stays in the flags file. A blank `ship_date` is **not** flagged: an unshipped order (e.g. `PLACED`) legitimately has none.

**Implement:** `parse_dates()` tries each format in turn on the still-unparsed values; flags `dq_order_date_unparsed`, `dq_ship_date_unparsed`, `dq_modified_at_unparsed`. The same day-first convention is applied to `ship_date`; `modified_at` also has a `dd-mm-yyyy HH:MM` form.

**Justify:** `dayfirst=True` guessing or `errors="coerce"` on a single format would either mis-read dates or null out ~7% of rows. Explicit formats parse everything that can be parsed unambiguously and make the one assumption visible. **[Unconfirmed]** day-first for `ship_date` (no file-level date to test against; checked only through the ship ≥ order rule in section 8).

In [6]:
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d", "%d-%m-%Y", "%b %d %Y"]
TS_FORMATS   = ["%Y-%m-%d %H:%M:%S", "%d-%m-%Y %H:%M"]

def parse_dates(s, formats):
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for fmt in formats:
        todo = out.isna() & s.notna()
        out[todo] = pd.to_datetime(s[todo], format=fmt, errors="coerce")
    return out

# Evidence for day-first: does the day in the date equal the day in the file name?
file_day = pd.to_datetime(df._source_file.str.extract(r"(\d{4}-\d{2}-\d{2})")[0])
for label, pat, fmt in [("dd/mm/yyyy", r"\d{2}/\d{2}/\d{4}", "%d/%m/%Y")]:
    m = df.order_date.str.fullmatch(pat, na=False)
    print(f"{label}: {int(m.sum())} rows, day-first equals file date in "
          f"{(pd.to_datetime(df.order_date[m], format=fmt, errors="coerce") == file_day[m]).mean():.0%}")

dd/mm/yyyy: 214 rows, day-first equals file date in 100%


In [7]:
df["order_date"]  = parse_dates(df["order_date"], DATE_FORMATS)
df["ship_date"]   = parse_dates(df["ship_date"],  DATE_FORMATS)
df["modified_at"] = parse_dates(df["modified_at"], TS_FORMATS)

# "unparsed" = there was a value (even a placeholder counts for order_date) but nothing usable came out
df["dq_order_date_unparsed"]    = df.order_date.isna()
df["dq_ship_date_unparsed"]     = df.ship_date.isna() & df_raw_values.raw_ship_date.str.strip().str.lower().ne("") & df_raw_values.raw_ship_date.notna()
df["dq_modified_at_unparsed"]   = df.modified_at.isna()
print(df[["dq_order_date_unparsed","dq_ship_date_unparsed","dq_modified_at_unparsed"]].sum().to_dict())

{'dq_order_date_unparsed': 41, 'dq_ship_date_unparsed': 31, 'dq_modified_at_unparsed': 6}


## 6. Numeric columns: `quantity`, `unit_price`, `amount`

**Identify:** `quantity` has non-integers (`two`, `3.0`; 29 rows). `amount` has currency symbols and thousands separators (`₹954.30`, `3,813.44`) and placeholders (`NaN`, `-`, `N/A`); 153 rows fail the decimal cast. 88 `amount` values are negative (report, Warn). `unit_price` is numeric everywhere; 25 `quantity` values are ≤ 0.

**Understand:** Totals, revenue and averages depend on these columns. Treating a null amount as 0 understates revenue; flipping a negative sign overstates it.

**Assess:** From the data: `₹` and `,` are pure formatting and removing them is lossless. `3.0` is a whole number, so it is safe as 3. `two` is a word, not a number we can read with a rule. Negative amounts have an interesting pattern: the check below shows how many equal exactly −(quantity × unit_price), i.e. a sign flip, but nothing in the report says *returns are stored negative*, so it could be a deliberate credit as easily as a defect.

**Decide:**
- `amount`: strip `₹` and `,`, cast to decimal; placeholders stay null (`dq_missing_amount`). Negative values are **kept as-is** and flagged (`dq_negative_amount`).
- `quantity`: accept integers and integer-valued decimals; `two` → null (`dq_invalid_quantity`); values ≤ 0 are kept and flagged (`dq_nonpositive_quantity`).
- `unit_price`: cast only; flag null or ≤ 0 (`dq_invalid_unit_price`).

**Implement:** vectorised regex clean + `to_numeric`; raw text stays in the flags file.

**Justify:** Zero-filling would silently change revenue; deriving `amount = quantity × unit_price` for nulls would be a new business rule; flipping signs would hide a possible upstream defect. Nulling the unreadable value and keeping the raw text preserves everything and lets analysts decide. **[Unconfirmed]** how negative amounts and a null amount should be treated (credit notes, or recompute from quantity × unit_price).

In [8]:
# --- amount ---
amt_txt = df["amount"].str.replace(r"[₹,\s]", "", regex=True)
df["amount"] = pd.to_numeric(amt_txt, errors="coerce").round(2)
df["dq_missing_amount"]  = df["amount"].isna()
df["dq_negative_amount"] = df["amount"] < 0

# --- quantity: integers and whole-number decimals only ---
q = pd.to_numeric(df["quantity"], errors="coerce")
q = q.where(q == q.round())                 # 3.0 ok, 2.5 would become null
df["quantity"] = q.astype("Int64")
df["dq_invalid_quantity"]     = df["quantity"].isna()
df["dq_nonpositive_quantity"] = df["quantity"] <= 0

# --- unit_price ---
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce").round(2)
df["dq_invalid_unit_price"] = df["unit_price"].isna() | (df["unit_price"] <= 0)

# Evidence for the negative-amount decision (concise): is it just a flipped sign?
neg = df[df.dq_negative_amount]
flipped = ((neg.amount + neg.quantity.astype(float) * neg.unit_price).abs() < 0.02).sum()
print(f"negative amounts: {len(neg)}, of which exactly -(quantity×unit_price): {int(flipped)}")
print(df[["dq_missing_amount","dq_invalid_quantity","dq_nonpositive_quantity","dq_invalid_unit_price"]].sum().to_dict())

negative amounts: 87, of which exactly -(quantity×unit_price): 86
{'dq_missing_amount': 64, 'dq_invalid_quantity': 12, 'dq_nonpositive_quantity': 26, 'dq_invalid_unit_price': 0}


## 7. Email, phone, status and text standardization

### 7a. Invalid emails
**Identify:** 642 emails fail the report's regex (12.26%, Block severity). **Understand:** Emails are contact data, not keys; an order with a bad email is still a sale. **Assess:** Beyond blanks and placeholders (handled in section 3), the failures are things like `test@`, a missing `@` (`manish.patelexample.com`) and upper-case addresses. The report's count is much higher than a plain structural check (about 250 failures) because it effectively requires ASCII-only addresses: an ASCII regex gives ~646 failures, almost exactly the report's 642, so ~390 of those are valid-looking addresses with accented letters such as `josé.singh@example.com`. **Decide:** lower-case the address; if it still has no valid `local@domain.tld` shape, set it to null and flag `dq_invalid_email`; **keep the order**. **Implement:** `lower()` + regex; raw email stays in the flags file. **Justify:** Dropping rows would lose orders for a contact-detail problem; "repairing" `manish.patelexample.com` would be guessing. Lower-casing is lossless because email domains are case-insensitive (and the local part is treated that way by virtually all providers). Non-ASCII local parts are kept as valid because they are structurally valid (RFC 6531 allows them), which is why our flag count is lower than the report's. **[Unconfirmed]** whether the stricter ASCII-only rule behind the report's count should apply.

### 7b. Unsupported `status`
**Identify:** 421 statuses are outside `{PLACED, SHIPPED, DELIVERED, CANCELLED, RETURNED}`. **Understand:** Status drives order-lifecycle reporting. **Assess:** Most failures are only case/padding (`cancelled`, `Shipped `) and are safely fixed by trimming and upper-casing. What remains (`UNKNWN`, `pending`, `?`, blank) is not in the approved set. `UNKNWN` is probably a typo for unknown and `pending` might mean `PLACED`, but nothing in the report supports a mapping. **Decide:** normalise case/whitespace; keep the unsupported ones as null in `status` (raw value preserved) with `dq_unsupported_status`. **Justify:** Mapping `pending → PLACED` would be an invented business rule; quarantining the row would remove otherwise valid orders. A null plus flag keeps the row and prevents it from being counted under the wrong status.

### 7c. Light text standardization (not in the report, evidence from the data)
`customer_name` values like `Aarav MenonÃ©` carry a trailing encoding artefact; `FARHAN IYER`/`Aarav Das ` vary in case and spacing; `city` varies in case/padding; `notes` contains tab characters; `phone` mixes `+91-`, spaces and the placeholder `0000000000`; `region_code` is empty in every row. We strip the `Ã©` suffix (the clean names elsewhere, e.g. `Zoë`, prove the suffix is not part of the name), title-case names and cities, collapse whitespace, and keep a 10-digit phone only when exactly 10 digits remain (placeholder all-zero → null). `region_code` stays as an all-null column so the schema is unchanged. We do **not** map `Bombay → Mumbai` or otherwise merge cities. **[Unconfirmed]**

In [9]:
# 7a email
df["email"] = df["email"].str.lower()
valid_email = df["email"].str.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s.]+", na=False)
df["dq_invalid_email"] = df["email"].notna() & ~valid_email
df.loc[df.dq_invalid_email, "email"] = pd.NA
df["dq_missing_email"] = df["email"].isna() & ~df["dq_invalid_email"]

# 7b status
st = df["status"].str.upper()
df["dq_unsupported_status"] = ~st.isin(ALLOWED_STATUS)
df["status"] = st.where(~df["dq_unsupported_status"])        # unsupported → null, raw kept in flags file

# 7c text columns
df["dq_name_repaired"] = df["customer_name"].str.contains("Ã©", na=False)
name = df["customer_name"].str.replace("Ã©", "", regex=False)
df["customer_name"] = name.str.replace(r"\s+", " ", regex=True).str.strip().str.title()
df["city"]  = df["city"].str.replace(r"\s+", " ", regex=True).str.strip().str.title()
df["notes"] = df["notes"].str.replace(r"\s+", " ", regex=True).str.strip()

digits = df["phone"].str.replace(r"\D", "", regex=True)
digits = digits.where(~digits.str.startswith("91", na=False) | (digits.str.len() != 12), digits.str[2:])   # drop +91 country code
df["phone"] = digits.where((digits.str.len() == 10) & (digits.str.strip("0") != ""))
df["dq_invalid_phone"] = df["phone"].isna() & df_raw_values.raw_phone.notna() & df_raw_values.raw_phone.str.strip().ne("")
df["region_code"] = df["region_code"].astype("object")

print(df[["dq_invalid_email","dq_missing_email","dq_unsupported_status","dq_name_repaired","dq_invalid_phone"]].sum().to_dict())
print(df.status.value_counts(dropna=False).to_dict())

{'dq_invalid_email': 174, 'dq_missing_email': 74, 'dq_unsupported_status': 47, 'dq_name_repaired': 114, 'dq_invalid_phone': 59}
{'CANCELLED': 1095, 'RETURNED': 1061, 'PLACED': 984, 'DELIVERED': 982, 'SHIPPED': 979, nan: 47}


## 8. Business-rule flag: `ship_date` earlier than `order_date`

**Identify:** 145 rows violate `ship_date ≥ order_date` (report, Warn severity; the report counted only ISO-formatted dates).

**Understand:** A shipment before the order is impossible, so one of the two dates is wrong; delivery-time metrics for these rows would be negative.

**Assess:** Neither the report nor the data tells us *which* date is wrong, so no repair is justified. Now that all dates are parsed we can recount over all rows.

**Decide:** Keep both dates unchanged and flag `dq_ship_before_order`. The count after parsing will be higher than 145 because mixed-format dates are now included; we report it rather than forcing a match.

**Justify:** Swapping dates or nulling one would be a guess. Flag-and-keep preserves both facts, and delivery-time analysis can exclude flagged rows.

In [10]:
df["dq_ship_before_order"] = (df["ship_date"] < df["order_date"]).fillna(False)
print("ship_date < order_date after full parsing:", int(df.dq_ship_before_order.sum()), "(report: 145 on ISO dates only)")

ship_date < order_date after full parsing: 164 (report: 145 on ISO dates only)


## 9. Duplicate business keys (`order_id` repeated with different content)

**Identify:** The report counts 130 duplicated `order_id` values (plus 29 empty keys) in source and Bronze.

**Understand:** If `order_id` is the grain of Silver, every repeated key double-counts an order.

**Assess:** After removing exact copies, repeated keys remain. Looking at examples, the rows usually differ in `status` and `modified_at` (e.g. the same order once as `PLACED` and once as `CANCELLED`), which looks like multiple *versions* of an order, but the report does not say whether the source is an event log or a snapshot.

**Decide:** Keep all rows; add `dq_duplicate_order_id` and `dq_order_id_rows` (how many rows share the key).

**Justify:** Picking the "latest" row by `modified_at` would be a defensible rule but it is not an established one: some duplicates have identical `modified_at` or placeholder timestamps, and a wrong pick loses the real status. Retaining all rows with a flag keeps the data complete and lets the winner rule be added once the grain is confirmed. **[Unconfirmed]** intended grain and survivor rule.

In [11]:
valid_key = df["order_id"].notna()
df["dq_order_id_rows"] = df.groupby("order_id")["order_id"].transform("size").where(valid_key)
df["dq_duplicate_order_id"] = df["dq_order_id_rows"].fillna(1) > 1
print("rows sharing an order_id:", int(df.dq_duplicate_order_id.sum()), "| distinct duplicated keys:", df.loc[df.dq_duplicate_order_id, "order_id"].nunique())

rows sharing an order_id: 116 | distinct duplicated keys: 58


## 10. Split into Silver and quarantine, and write outputs

**Decide / Implement:** Rows with no `order_id` leave the main flow (section 4) and go to quarantine with their raw line. Everything else goes to Silver. The Silver file contains only the 17 business columns (+ the trace key); all `dq_*` flags and raw values go to a separate flags file, keyed by `_source_file` + `_source_line`.

**Justify:** Keeping flags out of the business schema stops analysts from accidentally selecting technical columns and keeps Silver aligned with the intended schema, while the flags file still lets anyone filter, review or reverse any decision.

In [12]:
no_key = df["order_id"].isna()
for _, r in df[no_key].iterrows():
    quarantine.append(dict(_source_file=r._source_file, _source_line=r._source_line,
                           quarantine_reason="missing_order_id", raw_line=r._raw_line))
silver_df = df[~no_key].copy()

flag_cols = [c for c in silver_df.columns if c.startswith("dq_")]
silver_out = silver_df[COLS + ["_source_file", "_source_line"]].copy()
silver_out["order_date"]  = silver_out["order_date"].dt.strftime("%Y-%m-%d")
silver_out["ship_date"]   = silver_out["ship_date"].dt.strftime("%Y-%m-%d")
silver_out["modified_at"] = silver_out["modified_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

changed_raw = ["order_id","order_date","ship_date","status","quantity","amount","email","phone","customer_name","modified_at"]
flags_out = pd.concat([silver_df[["_source_file","_source_line"] + flag_cols],
                       df_raw_values.loc[silver_df.index, [f"raw_{c}" for c in changed_raw]],
                       silver_df[["_raw_line"]]], axis=1)
flags_out["dq_any_flag"] = flags_out[[c for c in flag_cols if c != "dq_order_id_rows"]].any(axis=1)

quarantine_out = pd.DataFrame(quarantine)[["_source_file","_source_line","quarantine_reason","raw_line"]]

silver_out.to_csv(OUT_DIR / "silver_orders.csv", index=False)
flags_out.to_csv(OUT_DIR / "silver_orders_dq_flags.csv", index=False)
quarantine_out.to_csv(OUT_DIR / "quarantine_orders.csv", index=False)

# Reconciliation: every input line is accounted for exactly once
print(f"input lines {total_lines} = silver {len(silver_out)} + quarantined {len(quarantine_out)}  →",
      total_lines == len(silver_out) + len(quarantine_out))
print(quarantine_out.quarantine_reason.value_counts().to_string())

input lines 5235 = silver 5095 + quarantined 140  → True
quarantine_reason
exact_duplicate_removed                         75
missing_order_id                                53
malformed_csv_9_fields_not_safely_repairable    12


In [13]:
# Outcome summary: how many Silver rows carry each flag
summary = (flags_out[[c for c in flag_cols if c != "dq_order_id_rows"] + ["dq_any_flag"]].sum()
           .astype(int).rename("rows").to_frame())
summary["pct_of_silver"] = (summary.rows / len(silver_out) * 100).round(2)
summary.sort_values("rows", ascending=False)

,rows,pct_of_silver
dq_any_flag,980,19.23
dq_invalid_email,171,3.36
dq_ship_before_order,162,3.18
dq_duplicate_order_id,116,2.28
dq_name_repaired,113,2.22
dq_negative_amount,86,1.69
dq_missing_email,73,1.43
dq_missing_amount,63,1.24
dq_invalid_phone,58,1.14
dq_unsupported_status,46,0.90


## Open business questions (nothing below was decided by assumption)

1. **Keyless orders** (no `order_id`): quarantined here; should they be kept in Silver instead?
2. **Grain and survivor rule** for repeated `order_id` with different content: all versions are kept and flagged until confirmed (e.g. latest `modified_at` wins?).
3. **Negative amounts / null amounts:** kept as-is; are negatives credit notes for returns, and may a null amount be recomputed from `quantity × unit_price`?
4. **Statuses** `pending`, `UNKNWN`, `?`, blank: left null and flagged; is there an approved mapping?
5. **Day-first dates** for `ship_date` and `modified_at` (verified against file dates only for `order_date`).
6. **Email rule:** should non-ASCII addresses count as invalid (the report's count suggests so)?
7. **City names** such as `Bombay` vs `Mumbai`, and the always-empty `region_code`.
8. **Ship date missing** for `SHIPPED`/`DELIVERED` orders is not flagged because the report has no such rule.